# 02.03 · Metrics, overfitting, cross-validation

**Chapter:** Machine Learning Fundamentals · **Milestone:** M1

## Goals
- [ ] Confusion matrix, precision, recall, F1, ROC-AUC — and when each matters
- [ ] Choose a decision threshold from the cost of errors
- [ ] Diagnose under/overfitting; use k-fold cross-validation

## Resources
- [scikit-learn – Model evaluation](https://scikit-learn.org/stable/modules/model_evaluation.html) — free
- [Google MLCC – Classification metrics](https://developers.google.com/machine-learning/crash-course/classification) — free

This lesson matters more than it looks: **chapter 10 (evals) is metrics applied to LLM systems.** An LLM-as-judge is a classifier;
a guardrail is a classifier; retrieval has precision and recall.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (ConfusionMatrixDisplay, RocCurveDisplay, classification_report,
                             f1_score, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures

rng = np.random.default_rng(0)

## 1. Imbalanced classification: accuracy lies
Fraud-like problem: 3% positives. Predicting "never fraud" gets 97% accuracy and catches nothing.

In [ ]:
X, y = make_classification(n_samples=5000, n_features=20, n_informative=6, weights=[0.97], flip_y=0.01, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
clf = LogisticRegression(max_iter=1000).fit(X_train, y_train)
p = clf.predict_proba(X_test)[:, 1]

print("always-negative accuracy:", round(1 - y_test.mean(), 3))
print(classification_report(y_test, p > 0.5, digits=3))
ConfusionMatrixDisplay.from_predictions(y_test, p > 0.5); plt.show()

- **Precision** = of what I flagged, how much was right? (TP / (TP+FP)) — matters when false alarms are costly.
- **Recall** = of all real positives, how many did I catch? (TP / (TP+FN)) — matters when misses are costly.
- **F1** = harmonic mean of the two.
- **ROC-AUC** = probability a random positive scores higher than a random negative. Threshold-independent ranking quality.

In [ ]:
RocCurveDisplay.from_predictions(y_test, p); plt.title(f"ROC-AUC = {roc_auc_score(y_test, p):.3f}"); plt.show()

## 2. The threshold is a product decision
0.5 is arbitrary. Sweep thresholds and pick by cost. Say a missed fraud costs \$100 and a false alarm costs \$5.

In [ ]:
thresholds = np.linspace(0.01, 0.99, 99)
prec = [precision_score(y_test, p > t, zero_division=0) for t in thresholds]
rec = [recall_score(y_test, p > t) for t in thresholds]
f1 = [f1_score(y_test, p > t) for t in thresholds]
cost = [100 * ((p <= t) & (y_test == 1)).sum() + 5 * ((p > t) & (y_test == 0)).sum() for t in thresholds]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.5))
ax1.plot(thresholds, prec, label="precision"); ax1.plot(thresholds, rec, label="recall"); ax1.plot(thresholds, f1, label="F1")
ax1.legend(); ax1.set_xlabel("threshold")
ax2.plot(thresholds, cost); ax2.set_xlabel("threshold"); ax2.set_title("total $ cost")
plt.tight_layout(); plt.show()
print(f"best-F1 threshold: {thresholds[np.argmax(f1)]:.2f} | cheapest threshold: {thresholds[np.argmin(cost)]:.2f}")

## 3. Underfitting vs overfitting (bias vs variance)
Fit polynomials of growing degree to noisy data. Training error keeps falling; validation error falls then rises.
The gap = overfitting (memorising noise).

In [ ]:
x = np.sort(rng.uniform(-3, 3, 60)); y_reg = np.sin(x) + rng.normal(0, 0.3, 60)
x_tr, x_va, y_tr, y_va = train_test_split(x, y_reg, test_size=0.4, random_state=0)
degrees = range(1, 16)
train_err, val_err = [], []
for d in degrees:
    m = make_pipeline(PolynomialFeatures(d), Ridge(alpha=1e-6)).fit(x_tr[:, None], y_tr)
    train_err.append(((m.predict(x_tr[:, None]) - y_tr) ** 2).mean())
    val_err.append(((m.predict(x_va[:, None]) - y_va) ** 2).mean())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.5))
ax1.plot(degrees, train_err, "o-", label="train"); ax1.plot(degrees, val_err, "o-", label="validation")
ax1.set_yscale("log"); ax1.set_xlabel("polynomial degree"); ax1.legend()
grid = np.linspace(-3, 3, 200)[:, None]
ax2.scatter(x_tr, y_tr, s=10)
for d in [1, 4, 15]:
    ax2.plot(grid, make_pipeline(PolynomialFeatures(d), Ridge(alpha=1e-6)).fit(x_tr[:, None], y_tr).predict(grid), label=f"deg {d}")
ax2.set_ylim(-2, 2); ax2.legend(); plt.tight_layout(); plt.show()

Fixes for overfitting: more data, simpler model, regularisation (L2, dropout), early stopping.
Deep nets and LLMs complicate the classic picture ("double descent"), but the train/val gap is still your main diagnostic.

## 4. k-fold cross-validation
With small data a single validation split is noisy. k-fold CV trains k times, each time validating on a different fold.
Report mean ± std — and notice how big the std is. Small eval sets give noisy numbers (same with LLM evals of 20 examples!).

In [ ]:
scores = cross_val_score(LogisticRegression(max_iter=1000), X, y, cv=KFold(5, shuffle=True, random_state=0), scoring="f1")
print("F1 per fold:", scores.round(3), f"| mean {scores.mean():.3f} ± {scores.std():.3f}")

## Exercises
1. Use `class_weight="balanced"` in `LogisticRegression`. How do precision/recall at 0.5 change? Why?
2. Add a regression-metrics section: MAE vs RMSE vs R² on the polynomial example. Which is most sensitive to outliers? Add 3 outliers and check.
3. Bootstrap a 95% confidence interval for test F1 (resample the test set 1000 times). How wide is it with 50 test examples vs 1500?
   Keep this in mind when someone claims "prompt B beats prompt A" on a tiny eval set.

In [ ]:
# your code here